# 48. Testing with pytest — `pytest`
**Python Libraries Lab · Phase 3 · Utilities & Testing**

_Simple, powerful tests that catch bugs before your users do._

⚙️ **Runs in Colab** after the setup cell below.

pytest is the most popular testing tool in Python. A test is just a function whose name starts with test, containing plain assert statements. pytest finds every test file, runs every test, and when one fails it shows you exactly which values were different. Teams run their tests before every change, so bugs are caught in seconds instead of by customers.

**In this notebook:** You will write a small ShopEasy pricing module, test it with plain asserts, add fixtures and parametrized cases, check that errors are raised correctly, then deliberately introduce a bug and watch pytest catch it.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q pytest
import importlib.metadata as md
for pkg in ['pytest']:
    print(pkg, md.version(pkg))

## Step 1 · Code to test
A small pricing module: cart totals, coupons and delivery fees.

In [ ]:
%%writefile pricing.py
def cart_total(items):
    """items: list of (price, qty)"""
    return sum(price * qty for price, qty in items)

def apply_coupon(total, code):
    coupons = {"SAVE10": 0.10, "FEST25": 0.25}
    if code is None:
        return total
    if code not in coupons:
        raise ValueError(f"Unknown coupon {code!r}")
    return round(total * (1 - coupons[code]), 2)

def delivery_fee(total):
    return 0 if total >= 999 else 49

## Step 2 · First tests
Each test is a function with `assert`. A fixture (`sample_cart`) provides shared data.

In [ ]:
%%writefile test_pricing.py
import pytest
from pricing import cart_total, apply_coupon, delivery_fee

@pytest.fixture
def sample_cart():
    return [(499, 2), (149, 3)]

def test_cart_total(sample_cart):
    assert cart_total(sample_cart) == 1445

def test_empty_cart():
    assert cart_total([]) == 0

def test_coupon(sample_cart):
    assert apply_coupon(cart_total(sample_cart), "SAVE10") == 1300.5

def test_no_coupon():
    assert apply_coupon(500, None) == 500

def test_unknown_coupon_raises():
    with pytest.raises(ValueError, match="Unknown coupon"):
        apply_coupon(500, "FREE100")

@pytest.mark.parametrize("total, fee", [(0, 49), (998, 49), (999, 0), (5000, 0)])
def test_delivery_fee(total, fee):
    assert delivery_fee(total) == fee

## Step 3 · Run the tests
`-v` lists each test. Parametrized cases appear separately.

In [ ]:
!python -m pytest test_pricing.py -v 2>&1 | tail -15

## Step 4 · Introduce a bug and watch it get caught
We change the free-delivery limit by mistake. pytest shows the exact failing values.

In [ ]:
from pathlib import Path
src = Path("pricing.py").read_text()
Path("pricing.py").write_text(src.replace("total >= 999", "total > 999"))
!python -m pytest test_pricing.py -q 2>&1 | tail -12
Path("pricing.py").write_text(src)     # fix it again

## Step 5 · Useful options
`-k` picks tests by name, `-x` stops at the first failure, `--durations` shows slow tests.

In [ ]:
!python -m pytest test_pricing.py -q -k "coupon"
!python -m pytest test_pricing.py -q -x --durations=3 2>&1 | tail -6

## ✍️ Your turn
Add a parametrized test for `apply_coupon` with `FEST25`: 1000 → 750.0 and 200 → 150.0. Write it to `test_more.py`, run pytest on it, and store the **number of passed tests** (an int) in `answer`. Hint: `out = !python -m pytest test_more.py -q --color=no` then read the last line.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 2, f"Expected 2 passed, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
Path("test_more.py").write_text("""import pytest
from pricing import apply_coupon

@pytest.mark.parametrize("total, expected", [(1000, 750.0), (200, 150.0)])
def test_fest25(total, expected):
    assert apply_coupon(total, "FEST25") == expected
""")
out = !python -m pytest test_more.py -q --color=no
answer = int(out[-1].split()[0])

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Test function**
- **assert**
- **Fixture**
- **parametrize**
- **pytest.raises**
- **Test discovery**

**Next:** 49. Pydantic Data Validation (`pydantic`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/49_pydantic.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install pytest
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.